# Juicio de expertos: coeficiente V de Aiken

Validez de contenido de las variables predictoras, las preguntas del formulario, los mensajes del resultado y el catálogo de recomendaciones (plan de tesis, 4.7).

- **Ítems:** `juicio_expertos/items.csv`, generados a partir de los textos del sistema.
- **Calificaciones:** `juicio_expertos/calificaciones.csv`. Una fila por ítem y criterio; una columna por experto (E1, E2, …), con valores de 1 a 4. Las columnas de expertos sin datos se ignoran.
- **Coeficiente:** V = Σ(xᵢ − 1) / [n (c − 1)], con n expertos y c = 4 categorías (Aiken, 1985). Intervalo de confianza del 95 % por el método de puntuación de Penfield y Giacobbi (2004).
- **Criterio de validez (plan de tesis):** V ≥ 0.80. Los ítems con V < 0.80 se corrigen; cada cambio se registra con fecha en el protocolo y la versión del catálogo se actualiza.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

ITEMS = Path('../../juicio_expertos/items.csv')
CALIFICACIONES = Path(r'../../juicio_expertos/calificaciones.csv')
SALIDA = Path(r'../../resultados/juicio_expertos')
C, Z, V_MINIMO = 4, 1.959964, 0.80

items = pd.read_csv(ITEMS)
cal = pd.read_csv(CALIFICACIONES)
expertos = [c for c in cal.columns if c.startswith('E') and cal[c].notna().any()]
print(f'{len(items)} ítems, {len(cal)} filas de calificación | expertos con datos: {expertos or "ninguno"}')

29 ítems, 93 filas de calificación | expertos con datos: ninguno


## Comprobaciones de los datos

In [2]:
HAY_DATOS = bool(expertos)
if not HAY_DATOS:
    print('La plantilla todavía no tiene calificaciones. Complete juicio_expertos/calificaciones.csv y vuelva a ejecutar el notebook.')
else:
    esperado = {(r.item_id, c) for r in items.itertuples() for c in r.criterios.split('|')}
    assert set(zip(cal.item_id, cal.criterio)) == esperado, 'las filas no coinciden con los ítems y criterios de items.csv'
    incompletos = {e: int(cal[e].isna().sum()) for e in expertos if cal[e].isna().any()}
    assert not incompletos, f'expertos con calificaciones faltantes: {incompletos}'
    valores = cal[expertos].to_numpy()
    assert np.isin(valores, [1, 2, 3, 4]).all(), 'hay calificaciones fuera de la escala de 1 a 4'
    print(f'Datos completos: {len(expertos)} expertos x {len(cal)} filas, todas en la escala 1-4.')

La plantilla todavía no tiene calificaciones. Complete juicio_expertos/calificaciones.csv y vuelva a ejecutar el notebook.


## V de Aiken por ítem y criterio

Límites del intervalo de Penfield y Giacobbi (2004), con k = c − 1:

L, U = [2nkV + z² ∓ z √(4nkV(1 − V) + z²)] / [2(nk + z²)]

In [3]:
def v_aiken(x):
    x = np.asarray(x, dtype=float)
    n, k = len(x), C - 1
    v = (x - 1).sum() / (n * k)
    raiz = Z * np.sqrt(4 * n * k * v * (1 - v) + Z ** 2)
    return v, (2 * n * k * v + Z ** 2 - raiz) / (2 * (n * k + Z ** 2)), (2 * n * k * v + Z ** 2 + raiz) / (2 * (n * k + Z ** 2))


if HAY_DATOS:
    filas = []
    for r in cal.itertuples(index=False):
        x = [getattr(r, e) for e in expertos]
        v, li, ls = v_aiken(x)
        filas.append({'item_id': r.item_id, 'criterio': r.criterio, 'media': np.mean(x), 'V': v, 'IC95_inf': li, 'IC95_sup': ls,
                      'valido (V >= 0.80)': 'Sí' if v >= V_MINIMO else 'No'})
    resultado = pd.DataFrame(filas).merge(items[['item_id', 'seccion', 'nombre']], on='item_id')
    resultado = resultado[['seccion', 'item_id', 'nombre', 'criterio', 'media', 'V', 'IC95_inf', 'IC95_sup', 'valido (V >= 0.80)']]
    SALIDA.mkdir(parents=True, exist_ok=True)
    resultado.to_csv(SALIDA / 'v_aiken_por_criterio.csv', index=False)
    pd.set_option('display.float_format', '{:.3f}'.format)
    display(resultado)

## Resumen por ítem y por sección

In [4]:
if HAY_DATOS:
    por_item = (resultado.groupby(['seccion', 'item_id', 'nombre'], sort=False)
                .agg(V_minimo=('V', 'min'), V_promedio=('V', 'mean'))
                .reset_index())
    por_item['valido (todos los criterios >= 0.80)'] = np.where(por_item['V_minimo'] >= V_MINIMO, 'Sí', 'No')
    por_item.to_csv(SALIDA / 'v_aiken_por_item.csv', index=False)
    display(por_item)
    por_seccion = resultado.groupby('seccion')['V'].agg(['mean', 'min']).rename(columns={'mean': 'V_promedio', 'min': 'V_minimo'})
    por_seccion.to_csv(SALIDA / 'v_aiken_por_seccion.csv')
    print(f"V de Aiken global: {resultado['V'].mean():.3f}")
    display(por_seccion)

## Ítems que requieren corrección

In [5]:
if HAY_DATOS:
    corregir = resultado[resultado['V'] < V_MINIMO]
    if corregir.empty:
        print('Todos los ítems alcanzan V >= 0.80 en todos los criterios.')
    else:
        print(f'{corregir.item_id.nunique()} ítems con algún criterio por debajo de 0.80. Revise las observaciones de los expertos en las fichas.')
        display(corregir)